In [ ]:
# Should be first to load all the env vars for browser
import os
from dotenv import load_dotenv

from browser import ChromeBrowser
from generalist.models.core import LLMBrowserWithTools, MLFlowLLMWrapper
from generalist.tools import WebSearchTool
from browser.search.web import BraveBrowser

load_dotenv()

assert os.getenv("CHROME_USER_DATA_DIR", None)
chrome_browser = ChromeBrowser()

In [ ]:
from generalist.agents.core import AgentDeepWebSearch
from generalist.tools.data_model import Message

import mlflow
import logging

mlflow.langchain.autolog()   # this is needed to register traces within the experiment
experiment_name = f"agent_tool_test"
mlflow.set_experiment(experiment_name)
logging.getLogger().setLevel(logging.INFO)

llm_raw = LLMBrowserWithTools(chrome_browser)
llm = MLFlowLLMWrapper(llm_instance=llm_raw)
# Includes access to the browser
brave_search_session = BraveBrowser(browser=chrome_browser, session_id="placeholder_brave_session")

https://stockanalysis.com/news/ & https://www.tradingview.com/news/

In [ ]:
content_resources = Message(
    provided_by= "user",
    content="",
    link= "",
)

task = "retrieve 3 ticker with the most market cap from https://stockanalysis.com/markets/active/"
agent = AgentDeepWebSearch(activity=task, llm=llm, brave_search_session=brave_search_session)
out = agent.run()

In [ ]:
from generalist.tools import ProcessTextFileTool

task = "Use www.tradingview.com to search INTC and summarise the page and then process the out with the task of 'Create a report summarising financial state and history of INTC'"
agent = AgentDeepWebSearch(
    activity=task,
    llm=llm,
    brave_search_session=brave_search_session,
    tools=[WebSearchTool(brave_search_session, llm), ProcessTextFileTool(llm=llm)]
)
out = agent.run()


In [ ]:
print(f"""
{agent.agent_state["context"][-1].content}
""")